# AKC-Tutor — Phase I: GoEmotions Emotion Recognition (Automatic Colab Experiment)

**Run all cells in order.** This notebook automatically downloads the official GoEmotions simplified splits, filters to **single-label** examples for a reproducible multiclass baseline, trains TF-IDF + Logistic Regression, evaluates on the held-out test set, produces figures and CSVs, and downloads a results ZIP.

**Scientific scope:** This is a **text emotion classification baseline**, not yet a validated affect-aware tutor. GoEmotions labels reflect Reddit comments, not verified student emotions. The single-label filter changes the evaluated population; results must not be described as full GoEmotions multi-label performance.

**Runtime:** CPU is sufficient. No GPU required. The notebook does not force-reinstall NumPy or alter Colab's preinstalled scientific stack.

## 1. Environment and reproducibility

In [ ]:
import sys, os, json, random, platform, subprocess
SEED = 42
random.seed(SEED)
import numpy as np
np.random.seed(SEED)
import pandas as pd
import scipy, sklearn, matplotlib, joblib
from pathlib import Path
from datetime import datetime, timezone
print('Python:', sys.version.split()[0]); print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__, 'sklearn:', sklearn.__version__)
print('Pandas:', pd.__version__, 'matplotlib:', matplotlib.__version__)
print('Runtime:', platform.platform())
OUTPUT = Path('/content/akc_tutor_phase1_results') if Path('/content').exists() else Path('akc_tutor_phase1_results')
OUTPUT.mkdir(parents=True, exist_ok=True)


## 2. Download official simplified GoEmotions files (automatic retries)

In [ ]:
import urllib.request, urllib.error, time
BASE = 'https://raw.githubusercontent.com/google-research/google-research/master/goemotions/data/'
DATA = OUTPUT / 'dataset'; DATA.mkdir(exist_ok=True)
for filename in ('train.tsv','dev.tsv','test.tsv','emotions.txt'):
    target = DATA / filename
    if target.exists() and target.stat().st_size > 0:
        print('Cached:', filename); continue
    last_error = None
    for attempt in range(4):
        try:
            req = urllib.request.Request(BASE+filename, headers={'User-Agent':'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=45) as r:
                content = r.read()
            if not content: raise ValueError('Empty download')
            target.write_bytes(content)
            print('Downloaded:', filename, len(content), 'bytes')
            break
        except Exception as exc:
            last_error = exc
            print(f'Attempt {attempt+1}/4 failed for {filename}: {exc}')
            time.sleep(2*(attempt+1))
    else:
        raise RuntimeError(f'Cannot download {filename}. Check Colab internet access. Last error: {last_error}')


## 3. Parse, validate, and prepare **single-label** classification data

In [ ]:
labels = [s.strip() for s in (DATA/'emotions.txt').read_text(encoding='utf-8').splitlines() if s.strip()]
assert len(labels) == len(set(labels)) and len(labels) >= 20, 'Unexpected emotion label file'

def load_split(name):
    # Simplified format: text, comma-separated emotion indices, comment id
    df = pd.read_csv(DATA/f'{name}.tsv', sep='\t', header=None, names=['text','label_ids','comment_id'],
                     dtype=str, keep_default_na=False, on_bad_lines='error')
    def parse_ids(raw):
        vals = [int(x.strip()) for x in raw.split(',') if x.strip()]
        if not vals or any(x < 0 or x >= len(labels) for x in vals):
            raise ValueError(f'Unexpected label ids: {raw!r}')
        return vals
    df['parsed_labels'] = df['label_ids'].map(parse_ids)
    original_n = len(df)
    df = df[df['parsed_labels'].map(len) == 1].copy()
    df['label_id'] = df['parsed_labels'].map(lambda x: x[0])
    df['emotion'] = df['label_id'].map(lambda x: labels[x])
    df = df[df['text'].str.strip().ne('')].reset_index(drop=True)
    print(f'{name}: {original_n:,} total; {len(df):,} single-label nonempty ({len(df)/original_n:.1%})')
    return df
train, dev, test = [load_split(x) for x in ('train','dev','test')]
# Detect any exact duplicated comment IDs between splits; never use test for tuning.
for a_name,a,b_name,b in [('train',train,'dev',dev),('train',train,'test',test),('dev',dev,'test',test)]:
    overlap = set(a.comment_id) & set(b.comment_id)
    print(f'Comment ID overlap {a_name}/{b_name}:',len(overlap))
    if overlap: raise ValueError(f'Split leakage: {len(overlap)} duplicate IDs between {a_name} and {b_name}')
counts = pd.DataFrame({'emotion':labels})
for name,df in [('train',train),('dev',dev),('test',test)]:
    counts[name] = counts.emotion.map(df.emotion.value_counts()).fillna(0).astype(int)
counts.to_csv(OUTPUT/'class_distribution.csv',index=False)
print(counts.to_string(index=False))
print('Number of labels:',len(labels))


## 4. Train baseline; select regularization on validation only

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

results = []
for c in [0.1, 1.0, 10.0]:
    model = Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=100000,
                                  sublinear_tf=True, strip_accents='unicode')),
        ('clf', LogisticRegression(C=c, max_iter=1000, solver='lbfgs', random_state=SEED))
    ])
    model.fit(train.text, train.label_id)
    dev_pred = model.predict(dev.text)
    score = f1_score(dev.label_id,dev_pred,labels=list(range(len(labels))),average='macro',zero_division=0)
    results.append({'C':c,'validation_macro_f1':float(score)})
    print('C=',c,'validation Macro-F1=',round(score,4))
validation = pd.DataFrame(results).sort_values(['validation_macro_f1','C'],ascending=[False,True])
validation.to_csv(OUTPUT/'validation_hyperparameters.csv',index=False)
best_c = float(validation.iloc[0]['C'])
print('Selected C:',best_c)
# Refit on train + dev after selecting C, with untouched test split.
train_dev = pd.concat([train,dev],ignore_index=True)
final_model = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=100000,
                              sublinear_tf=True, strip_accents='unicode')),
    ('clf', LogisticRegression(C=best_c,max_iter=1000,solver='lbfgs',random_state=SEED))
])
final_model.fit(train_dev.text,train_dev.label_id)
joblib.dump(final_model,OUTPUT/'tfidf_logreg_model.joblib',compress=3)
print('Model saved.')


## 5. Held-out test evaluation and predictions

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt
pred = final_model.predict(test.text)
y_true = test.label_id.to_numpy()
ids = list(range(len(labels)))
report = classification_report(y_true,pred,labels=ids,target_names=labels,output_dict=True,zero_division=0)
metrics = {
    'accuracy':float(accuracy_score(y_true,pred)),
    'macro_f1':float(f1_score(y_true,pred,labels=ids,average='macro',zero_division=0)),
    'weighted_f1':float(f1_score(y_true,pred,labels=ids,average='weighted',zero_division=0)),
    'n_train':int(len(train)),'n_dev':int(len(dev)),'n_test':int(len(test)),
    'best_C':best_c,'seed':SEED,'label_count':len(labels),
    'data_subset':'single-label rows of official simplified GoEmotions',
    'test_used_for_tuning':False
}
print(json.dumps(metrics,indent=2))
(OUTPUT/'test_metrics.json').write_text(json.dumps(metrics,indent=2))
pd.DataFrame(report).T.to_csv(OUTPUT/'classification_report.csv')
pd.DataFrame({'comment_id':test.comment_id,'text':test.text,
              'true_label':test.emotion,'predicted_label':[labels[int(i)] for i in pred]}).to_csv(OUTPUT/'test_predictions.csv',index=False)
cm = confusion_matrix(y_true,pred,labels=ids)
pd.DataFrame(cm,index=labels,columns=labels).to_csv(OUTPUT/'confusion_matrix.csv')
fig,ax = plt.subplots(figsize=(14,12))
ConfusionMatrixDisplay(cm,display_labels=labels).plot(ax=ax,xticks_rotation=90,include_values=False,cmap='Blues',colorbar=True)
ax.set_title('GoEmotions Single-label Test Confusion Matrix')
fig.tight_layout();fig.savefig(OUTPUT/'confusion_matrix.png',dpi=180);plt.show();plt.close(fig)


## 6. Bootstrap 95% confidence interval for test Macro-F1

In [ ]:
rng = np.random.default_rng(SEED)
B = 1000
bootstrap_f1 = np.empty(B)
for i in range(B):
    sample = rng.integers(0,len(y_true),size=len(y_true))
    bootstrap_f1[i] = f1_score(y_true[sample],pred[sample],labels=ids,average='macro',zero_division=0)
ci_low,ci_high = np.quantile(bootstrap_f1,[0.025,0.975])
metrics['macro_f1_bootstrap_95ci']=[float(ci_low),float(ci_high)]
metrics['bootstrap_replicates']=B
(OUTPUT/'test_metrics.json').write_text(json.dumps(metrics,indent=2))
print(f"Test Macro-F1: {metrics['macro_f1']:.4f} (95% bootstrap CI {ci_low:.4f}–{ci_high:.4f})")
fig,ax=plt.subplots(figsize=(8,4))
ax.hist(bootstrap_f1,bins=35)
ax.axvline(ci_low,linestyle='--',label='95% lower')
ax.axvline(ci_high,linestyle='--',label='95% upper')
ax.set(xlabel='Bootstrap Macro-F1',ylabel='Frequency',title='Bootstrap Distribution of Test Macro-F1')
ax.legend();fig.tight_layout();fig.savefig(OUTPUT/'macro_f1_bootstrap.png',dpi=180);plt.show();plt.close(fig)


## 7. Save run metadata and download ZIP automatically

In [ ]:
import shutil
metadata = {
    'timestamp_utc':datetime.now(timezone.utc).isoformat(),
    'python':sys.version,'numpy':np.__version__,'scipy':scipy.__version__,
    'sklearn':sklearn.__version__,'pandas':pd.__version__,
    'dataset_url':BASE,'experiment':'TF-IDF + logistic regression, single-label subset',
    'method':'validation-based C selection, refit on train+dev, held-out test',
    'limitations':['GoEmotions is Reddit-derived, not learner-specific',
                   'Multilabel rows excluded','No tutor effectiveness evaluation']
}
(OUTPUT/'run_metadata.json').write_text(json.dumps(metadata,indent=2))
(OUTPUT/'README.txt').write_text('AKC-Tutor Phase I baseline. See test_metrics.json, classification_report.csv, and run_metadata.json.\n')
# Do not bundle raw dataset or test comment texts: package results and predictions separately.
zip_base = OUTPUT.parent/'AKC_Tutor_Phase1_Results'
zip_path = shutil.make_archive(str(zip_base),'zip',root_dir=OUTPUT,base_dir=None)
print('Saved:',zip_path)
print('Size (MB):',round(Path(zip_path).stat().st_size/1024**2,2))
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print('Not running in Colab; ZIP is available at:',zip_path)


## Next research step
Upload `AKC_Tutor_Phase1_Results.zip` to this conversation. We will inspect **measured** performance, error patterns, and class imbalance before moving to a RoBERTa baseline. Avoid claiming that emotion classification alone proves improved tutoring or learning outcomes.